<a href="https://colab.research.google.com/github/Zain708/DevelopersHub-AI-ML-Internship-Advanced-Tasks-Zainab/blob/main/Task_1_BERT_News_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install Hugging Face Transformers, Datasets, and Accelerate for training
!pip install -q transformers[torch] datasets evaluate scikit-learn gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.8 MB/s eta 0:00:00


In [ ]:
import torch
print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))

GPU Available: True
Device Name: Tesla T4


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

# 1. Load the AG News Dataset using the stable namespace path
print("Loading AG News dataset...")
dataset = load_dataset("fancyzhx/ag_news")

# 2. Downsample for fast training (keeps you moving on your timeline!)
train_dataset = dataset["train"].shuffle(seed=42).select(range(10000))
test_dataset = dataset["test"].shuffle(seed=42).select(range(2000))

# 3. Load the BERT Tokenizer
print("Loading bert-base-uncased tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# 4. Define tokenization function
# Since 'text' already contains the news article, we can tokenize it directly!
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

# 5. Tokenize the datasets
print("Tokenizing datasets...")
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

print("Preprocessing complete!")

Loading AG News dataset...


README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizing datasets...


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Preprocessing complete!


In [2]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.3 MB/s eta 0:00:00


In [4]:
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

# 1. Load and Tokenize Dataset
print("Loading and tokenizing dataset...")
dataset = load_dataset("fancyzhx/ag_news")
train_dataset = dataset["train"].shuffle(seed=42).select(range(10000))
test_dataset = dataset["test"].shuffle(seed=42).select(range(2000))

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

# 2. Setup Metrics
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="macro")["f1"]
    return {"accuracy": acc, "f1": f1}

# 3. Load Model and Train
print("Loading BERT model...")
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=4)

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=1,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_dir="./logs",
    logging_steps=100,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

print("Starting training on GPU...")
trainer.train()

# 4. Save Model Locally
print("Saving fine-tuned model...")
model.save_pretrained("./fine_tuned_bert")
tokenizer.save_pretrained("./fine_tuned_bert")
print("Model and tokenizer successfully saved!")

Loading and tokenizing dataset...


README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Saving fine-tuned model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer successfully saved!


In [5]:
# Save the model and tokenizer to a local directory
print("Saving fine-tuned model...")
model.save_pretrained("./fine_tuned_bert")
tokenizer.save_pretrained("./fine_tuned_bert")
print("Model and tokenizer successfully saved!")

Saving fine-tuned model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer successfully saved!


In [6]:
import gradio as gr
from transformers import pipeline

# 1. Map the model's numerical labels to the actual AG News categories
category_map = {
    "LABEL_0": "World",
    "LABEL_1": "Sports",
    "LABEL_2": "Business",
    "LABEL_3": "Sci/Tech"
}

# 2. Load your saved model into a Hugging Face pipeline
# device=0 ensures it runs lightning fast on your T4 GPU
print("Loading model for deployment...")
classifier = pipeline("text-classification", model="./fine_tuned_bert", tokenizer="./fine_tuned_bert", device=0)

# 3. Define the prediction function
def predict_topic(news_text):
    result = classifier(news_text)[0]
    predicted_label = result['label']
    confidence = result['score']

    category = category_map.get(predicted_label, "Unknown")
    return f"Predicted Category: {category}\nConfidence Score: {confidence:.2%}"

# 4. Build the Gradio Web Interface
interface = gr.Interface(
    fn=predict_topic,
    inputs=gr.Textbox(lines=4, placeholder="Paste a news headline or short article here..."),
    outputs=gr.Textbox(label="Classification Result"),
    title="News Topic Classifier Using BERT",
    description="Enter a news headline to classify it into World, Sports, Business, or Sci/Tech categories."
)

# 5. Launch the app and generate a public share link
print("Launching Gradio web app...")
interface.launch(share=True)

Loading model for deployment...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Launching Gradio web app...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://41ccb315ece6148dbc.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
